<a href="https://colab.research.google.com/github/2632725-stack/DES-Algorithm/blob/main/Implementation_of_AES_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
# AES (Advanced Encryption Standard) implementation
#AES-128: Uses a 128-bit key (10 encryption rounds).
# By Rajat Chowdhury
# Id 2632725
# M.Sc. in Computer Networks and Communications, IUB

In [54]:
# NIST FIPS-197 Standard Constants
S_BOX = [
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16
]
RCON = [0x00, 0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]

def bytes2matrix(text):
    return [list(text[i:i+4]) for i in range(0, len(text), 4)]

def matrix2bytes(matrix):
    return bytes(sum(matrix, []))

def xor_matrices(a, b):
    return [[a[i][j] ^ b[i][j] for j in range(4)] for i in range(4)]

def multiply_gf28(a, b):
    p = 0
    for _ in range(8):
        if b & 1: p ^= a
        hi_bit_set = a & 0x80
        a <<= 1
        if hi_bit_set: a ^= 0x1B
        b >>= 1
    return p & 0xFF

print("[SUCCESS] Global variables and mathematical helper functions are loaded.")


[SUCCESS] Global variables and mathematical helper functions are loaded.


In [55]:
# Target 16-byte text input and 16-byte key configuration
plaintext_block = b"RajatChowdhuryee"
master_key = b"MySecretKey128Bi"

# Map to 4x4 State Matrix
state = bytes2matrix(plaintext_block)
print("[SUCCESS] Step 1: Input mapped into 4x4 State Matrix successfully.")
print("Current State Matrix structure generated.")


[SUCCESS] Step 1: Input mapped into 4x4 State Matrix successfully.
Current State Matrix structure generated.


In [56]:
def expand_key(key_source):
    key_symbols = list(key_source)
    while len(key_symbols) < 176:
        last_word = key_symbols[-4:]
        if len(key_symbols) % 16 == 0:
            last_word = last_word[1:] + last_word[:1]
            last_word = [S_BOX[b] for b in last_word]
            last_word[0] ^= RCON[len(key_symbols) // 16]
        prev_word = key_symbols[-16:-12]
        for i in range(4):
            key_symbols.append(prev_word[i] ^ last_word[i])
    return [bytes2matrix(key_symbols[i:i+16]) for i in range(0, 176, 16)]

# Execute extension
round_keys = expand_key(master_key)
print("[SUCCESS] Step 2: Key Expansion complete. 11 Round Keys generated.")

[SUCCESS] Step 2: Key Expansion complete. 11 Round Keys generated.


In [57]:
# XOR state matrix with Round Key 0
state = xor_matrices(state, round_keys[0])
print("[SUCCESS] Step 3: Initial AddRoundKey operation done.")

[SUCCESS] Step 3: Initial AddRoundKey operation done.


In [58]:
def sub_bytes(current_state):
    return [[S_BOX[b] for b in row] for row in current_state]

def shift_rows(current_state):
    new_state = [[0 for _ in range(4)] for _ in range(4)]
    # Row 0: no shift
    new_state[0] = current_state[0][:] # Make a copy

    # Row 1: circular shift left by 1
    new_state[1] = current_state[1][1:] + current_state[1][:1]

    # Row 2: circular shift left by 2
    new_state[2] = current_state[2][2:] + current_state[2][:2]

    # Row 3: circular shift left by 3
    new_state[3] = current_state[3][3:] + current_state[3][:3]
    return new_state

def mix_columns(current_state):
    new_state = [[0 for _ in range(4)] for _ in range(4)] # Correct initialization as 4x4 matrix

    # Fixed matrix for MixColumns
    # 02 03 01 01
    # 01 02 03 01
    # 01 01 02 03
    # 03 01 01 02

    for c in range(4): # Iterate through columns
        s0 = current_state[0][c]
        s1 = current_state[1][c]
        s2 = current_state[2][c]
        s3 = current_state[3][c]

        new_state[0][c] = multiply_gf28(0x02, s0) ^ multiply_gf28(0x03, s1) ^ s2 ^ s3
        new_state[1][c] = s0 ^ multiply_gf28(0x02, s1) ^ multiply_gf28(0x03, s2) ^ s3
        new_state[2][c] = s0 ^ s1 ^ multiply_gf28(0x02, s2) ^ multiply_gf28(0x03, s3)
        new_state[3][c] = multiply_gf28(0x03, s0) ^ s1 ^ s2 ^ multiply_gf28(0x02, s3)
    return new_state

# Loop for Round 1 to 9
for r in range(1, 10):
    state = sub_bytes(state)      # 4.1 SubBytes
    state = shift_rows(state)     # 4.2 ShiftRows
    state = mix_columns(state)    # 4.3 MixColumns
    state = xor_matrices(state, round_keys[r]) # 4.4 AddRoundKey

print("[SUCCESS] Step 4: Round 1 to 9 main processing loop complete.")

[SUCCESS] Step 4: Round 1 to 9 main processing loop complete.


In [59]:
# Execute final round operations
state = sub_bytes(state)      # 5.1 SubBytes
state = shift_rows(state)     # 5.2 ShiftRows
state = xor_matrices(state, round_keys[10]) # 5.3 AddRoundKey (Final Key)

print("[SUCCESS] Step 5: Round 10 complete (MixColumns skipped).")

[SUCCESS] Step 5: Round 10 complete (MixColumns skipped).


In [60]:
# Convert final matrix block back to continuous bytes
ciphertext = matrix2bytes(state)

print("="*60)
print(f"[SUCCESS] Step 6: Final Ciphertext (Hex Code):\n{ciphertext.hex()}")
print("="*60)


[SUCCESS] Step 6: Final Ciphertext (Hex Code):
53c717542fd31596f930e9e99bf83061
